# Notebook 04 — BETO: Embeddings Contextuales
**Proyecto 2 | Análisis Semántico de Letras Musicales**  
**Curso:** Minería de Textos — CUC  

---
## Objetivo
Usar BETO (Spanish BERT) para generar embeddings contextuales, demostrar polisemia,
implementar búsqueda semántica y analizar el Masked Language Model por género.


## 0. Dependencias

In [ ]:
!pip install -q transformers torch scikit-learn matplotlib seaborn pandas numpy pymongo python-dotenv
print("Dependencias listas")

## 1. Cargar corpus desde MongoDB

In [ ]:
import sys
sys.path.append("..")

from src.entities.consultar_base_datos import consultar_base_datos

cargador = consultar_base_datos()
cargador.cargar_todas()
df = cargador.df

print(f"Canciones cargadas: {len(df):,}")
print(f"Géneros: {df['genero'].value_counts().to_dict()}")

## 2. Cargar BETO desde HuggingFace

In [ ]:
from src.embeddings.embeddings_beto import CargadorBETO

beto = CargadorBETO()
tokenizer = beto.tokenizer
model     = beto.model

## 3. Polisemia contextual: la misma palabra, vectores distintos

Word2Vec asigna **un único vector** por palabra.  
BETO asigna vectores **distintos** según el contexto que rodea a esa palabra.


In [ ]:
from src.embeddings.embeddings_beto import analizar_polisemia, embedding_token
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Palabra polisémica 1: "heart" (corazón / parte central) ──
oraciones_heart = [
    ("I can feel your heart beating in the dark", "heart"),
    ("The heart of the problem is poverty", "heart"),
    ("She gave her heart to the music", "heart"),
]

print("=== Polisemia de 'heart' ===\n")
embeddings = []
for texto, pal in oraciones_heart:
    emb, tokens = embedding_token(texto, pal, tokenizer, model)
    if emb is not None:
        embeddings.append(emb)
        print(f"Contexto: {texto}")
        print(f"Tokens:   {tokens[:10]}")
        print(f"Vector primeros 5: {emb[:5].round(4)}\n")

if len(embeddings) >= 2:
    for i in range(len(embeddings)):
        for j in range(i+1, len(embeddings)):
            sim = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]
            print(f"  Similitud ctx{i+1} vs ctx{j+1}: {sim:.4f}")

In [ ]:
# ── Palabra polisémica 2: "fire" (fuego / despedir / disparar) ──
oraciones_fire = [
    ("There is fire in her eyes when she sings", "fire"),
    ("They set the building on fire last night", "fire"),
    ("The boss decided to fire him after the incident", "fire"),
]

print("=== Polisemia de 'fire' ===\n")
embs_fire = []
for texto, pal in oraciones_fire:
    emb, _ = embedding_token(texto, pal, tokenizer, model)
    if emb is not None:
        embs_fire.append(emb)
        print(f"Contexto: {texto}")

if len(embs_fire) >= 3:
    import pandas as pd
    nombres = ["fire(metáfora)", "fire(literal)", "fire(despedir)"]
    matriz_sim = [[cosine_similarity([embs_fire[i]], [embs_fire[j]])[0][0]
                   for j in range(len(embs_fire))]
                  for i in range(len(embs_fire))]
    df_sim_poli = pd.DataFrame(matriz_sim, index=nombres, columns=nombres)
    print("\nMatriz de similitud contextual:")
    print(df_sim_poli.round(4).to_string())

In [ ]:
# Seleccionar 5 palabras polisémicas del corpus musical y documentar
PALABRAS_POLISEMICAS = {
    "rock":  ["I love rock and roll music", "The rock fell from the mountain", "She sat on a rock by the river"],
    "beat":  ["I feel the beat of your heart", "He beat the drum all night", "They beat the team in the finals"],
    "break": ["My heart is going to break tonight", "Take a break before the show", "Break the chains that hold you down"],
    "light": ["She is the light of my life", "The light at the end of the tunnel", "Turn off the light before you leave"],
    "fall":  ["Baby don't let me fall", "Leaves fall in the autumn wind", "The empire will fall"],
}

resultados_polisemia = {}
for palabra, contextos in PALABRAS_POLISEMICAS.items():
    embs = []
    print(f"\n=== '{palabra}' ===")
    for texto in contextos:
        emb, _ = embedding_token(texto, palabra, tokenizer, model)
        if emb is not None:
            embs.append(emb)
    if len(embs) >= 2:
        sims = [cosine_similarity([embs[i]], [embs[j]])[0][0]
                for i in range(len(embs)) for j in range(i+1, len(embs))]
        resultados_polisemia[palabra] = {
            "n_contextos": len(embs),
            "similitud_promedio": round(float(np.mean(sims)), 4),
            "similitud_min": round(float(np.min(sims)), 4),
        }
        print(f"  Similitud promedio entre contextos: {np.mean(sims):.4f}")
        print(f"  Similitud mínima: {np.min(sims):.4f} (más diferente semánticamente)")

## 4. Embeddings de canciones completas (vector [CLS])

In [ ]:
from src.embeddings.embeddings_beto import embedding_cls
import numpy as np

# Trabajar con una muestra para no saturar memoria/tiempo
MUESTRA = min(500, len(df))
df_muestra = df.sample(MUESTRA, random_state=42).reset_index(drop=True)
textos = df_muestra["letra"].fillna("").tolist()

print(f"Generando embeddings BETO [CLS] para {MUESTRA} canciones...")
beto_embeddings = embedding_cls(textos, tokenizer, model, batch_size=16)
print(f"Matriz resultante: {beto_embeddings.shape}")

## 5. Búsqueda semántica de canciones

In [ ]:
from src.embeddings.embeddings_beto import BuscadorSemantico

buscador = BuscadorSemantico(df_muestra, tokenizer, model,
                             col_titulo="titulo", col_artista="artista",
                             col_genero="genero", col_letra="letra")
buscador.indexar()

In [ ]:
# Buscar canciones similares a distintas consultas
consultas = [
    "dancing under the moonlight feeling free",
    "broken heart crying alone in the dark",
    "fight for your rights stand up against the system",
    "love is all you need feel the warmth",
    "speed and adrenaline wild night out party",
]

for consulta in consultas:
    print("\n" + "="*60)
    buscador.buscar(consulta, top_k=5)

## 6. Masked Language Model por género

In [ ]:
from src.embeddings.embeddings_beto import AnalizadorMLM

mlm = AnalizadorMLM()

In [ ]:
# Frases con [MASK] representativas de letras musicales
# BETO usa [MASK], no <mask>
PLANTILLAS = [
    "I can feel your [MASK] beating in the dark",
    "We dance all [MASK] long to the rhythm",
    "My heart is [MASK] without you here",
    "The crowd goes [MASK] when the beat drops",
]

print("=== Predicciones generales de BETO ===")
for plantilla in PLANTILLAS:
    mlm.predecir(plantilla, top_k=5)

In [ ]:
# Palabras más frecuentes por género para construir plantillas contextualizadas
frecuentes = mlm.palabras_frecuentes_por_genero(
    df, col_genero="genero", col_letra="letra", top_n=20
)

# Plantillas usando contexto de cada género
print("\n=== Análisis MLM por género ===")
for genero, top_words in frecuentes.items():
    palabras_top = [w for w, _ in top_words[:5]]
    print(f"\nGénero: {genero} | Palabras más frecuentes: {palabras_top}")
    
    # Crear plantilla con contexto del género
    if palabras_top:
        plantilla = f"I feel the [MASK] of {palabras_top[0]}"
        print(f"  Plantilla: {plantilla}")
        mlm.predecir(plantilla, top_k=5)

## 7. Visualización t-SNE de canciones con BETO

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(beto_embeddings)

colores_genero = {
    "Rock": "#e74c3c", "Pop": "#3498db", "Hip-Hop": "#2ecc71",
    "Reggaetón": "#f39c12", "Balada": "#9b59b6"
}

generos_muestra = df_muestra["genero"].fillna("Unknown").tolist()
generos_unicos  = list(set(generos_muestra))

fig, ax = plt.subplots(figsize=(12, 9))
for genero in generos_unicos:
    idx = [i for i, g in enumerate(generos_muestra) if g == genero]
    color = colores_genero.get(genero, "#95a5a6")
    ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1], c=color,
               label=genero, s=40, alpha=0.65, edgecolors="none")

ax.set_title("Canciones en espacio BETO — proyección t-SNE",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, markerscale=2)
ax.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig("../docs/tsne_beto_canciones.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Actualizar embeddings en MongoDB

In [ ]:
from src.embeddings.embeddings_beto import actualizar_beto_cls_mongodb

# Actualiza embeddings.beto_cls para toda la colección
actualizar_beto_cls_mongodb(
    df=df,
    tokenizer=tokenizer,
    model=model,
    col_id="id",
    col_letra="letra",
    batch_size=32,
)
print("beto_cls actualizado en MongoDB.")

## 9. Diferencias clave: Word2Vec vs BETO

| Aspecto | Word2Vec | BETO |
|---|---|---|
| Tipo de representación | Estática (1 vector / palabra) | Contextual (vector distinto según contexto) |
| Polisemia | No distingue significados | Vectores distintos para cada uso |
| Dimensión | 100–300d | 768d |
| Velocidad de inferencia | Muy rápida | Lenta (necesita GPU para corpus grandes) |
| Entrenamiento propio | Sí, sobre el corpus propio | No (se usa modelo pre-entrenado) |
| Mejor para | Similitud léxica, analogías sencillas | Comprensión profunda, búsqueda semántica |

**Cuándo usar BETO:** cuando la misma palabra tiene significados distintos según el contexto,
o cuando se necesita capturar el significado de frases completas (búsqueda semántica, clasificación).
